In [ ]:
```xml
<VSCode.Cell language="markdown">
# 🔒 Data Privacy & GDPR Compliance Guide

**Building privacy-first systems that protect user data and ensure regulatory compliance**

This guide covers:
- GDPR core principles and implementation
- Data protection by design
- Privacy-preserving architectures
- Encryption and data security
- Data retention and deletion policies
- Privacy impact assessments
- Incident response
- International regulations (GDPR, CCPA, PIPEDA)
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. GDPR Core Principles

### Seven Pillars of Data Protection

**1. Lawfulness, Fairness, Transparency**
```python
# Document legal basis for data processing
class DataProcessing:
    """Track lawful basis for data collection"""
    
    LEGAL_BASIS = {
        'consent': 'User explicitly consented',
        'contract': 'Necessary for service delivery',
        'legal_obligation': 'Required by law',
        'vital_interest': 'Protect human health/life',
        'public_task': 'Official government function',
        'legitimate_interest': 'Business interests (balanced)'
    }
    
    def __init__(self, data_type: str, basis: str, description: str):
        self.data_type = data_type
        self.basis = basis
        self.description = description
        self.collection_date = datetime.now()
        
        # Log processing
        self._log_processing()
    
    def _log_processing(self):
        """Maintain audit trail"""
        log_entry = {
            'timestamp': self.collection_date,
            'data_type': self.data_type,
            'legal_basis': self.basis,
            'description': self.description
        }
        self._store_audit_log(log_entry)

# Usage
consent_form = DataProcessing(
    data_type='email',
    basis='consent',
    description='User opted in to marketing emails'
)
```

**2. Purpose Limitation**
```python
# Only use data for stated purposes
class PurposeLimitation:
    PURPOSES = {
        'marketing': ['email_marketing', 'sms_marketing', 'push_notifications'],
        'service': ['account_management', 'feature_delivery', 'support'],
        'analytics': ['usage_analytics', 'product_improvement'],
        'security': ['fraud_detection', 'account_security']
    }
    
    async def validate_usage(self, data_type: str, purpose: str) -> bool:
        """Check if data can be used for purpose"""
        if purpose not in self.PURPOSES:
            raise ValueError(f"Invalid purpose: {purpose}")
        
        allowed = await self.get_allowed_purposes(data_type)
        
        if purpose not in allowed:
            logger.warning(f"Attempted misuse: {data_type} for {purpose}")
            return False
        
        return True
    
    async def get_allowed_purposes(self, data_type: str) -> List[str]:
        """Get approved purposes for data type"""
        # Query privacy settings/consent records
        return await db.query(
            "SELECT purposes FROM user_consents WHERE data_type = ?",
            (data_type,)
        )
```

**3. Data Minimization**
```typescript
// Only collect necessary data
interface UserRegistration {
  // Mandatory
  email: string;
  password_hash: string;
  
  // Optional (ask explicit permission)
  first_name?: string;
  last_name?: string;
  phone?: string;
  
  // NEVER collect unless needed
  // social_security_number: never;
  // financial_records: never;
}

// Validate data minimization
function validateDataMinimization(userData: any): boolean {
  const allowed_fields = ['email', 'password_hash', 'first_name', 'last_name', 'phone'];
  
  for (const field of Object.keys(userData)) {
    if (!allowed_fields.includes(field)) {
      logger.warn(`Unnecessary field collected: ${field}`);
      delete userData[field];
    }
  }
  
  return true;
}
```

**4. Accuracy**
```python
# Maintain data quality and accuracy
class DataAccuracy:
    async def update_user_data(self, user_id: str, data: dict):
        """Update with audit trail"""
        old_data = await get_user_data(user_id)
        
        # Track changes
        for key, new_value in data.items():
            old_value = old_data.get(key)
            if old_value != new_value:
                await log_data_change(
                    user_id=user_id,
                    field=key,
                    old_value=old_value,
                    new_value=new_value,
                    timestamp=datetime.now()
                )
        
        # Update record
        await update_user(user_id, data)
    
    async def request_data_correction(self, user_id: str, corrections: dict):
        """User requests data correction"""
        # Validate corrections
        for field, new_value in corrections.items():
            if field in ['email', 'name', 'address']:
                # Require verification for sensitive fields
                verification_token = self._generate_verification_token()
                await send_verification_email(
                    user_id=user_id,
                    changes=corrections,
                    token=verification_token
                )
                return {'status': 'verification_required'}
        
        # Update data
        await self.update_user_data(user_id, corrections)
        return {'status': 'updated'}
```

**5. Storage Limitation**
```python
# Automatic data retention and deletion
class DataRetention:
    RETENTION_POLICIES = {
        'user_profile': 'lifetime',
        'activity_logs': 90,        # days
        'payment_records': 2555,    # 7 years (tax requirement)
        'marketing_consent': 365,   # 1 year
        'failed_login_attempts': 30 # 30 days
    }
    
    async def schedule_deletion(self, data_type: str, user_id: str):
        """Schedule automatic deletion"""
        retention_days = self.RETENTION_POLICIES.get(data_type)
        
        if retention_days == 'lifetime':
            return  # Keep forever
        
        deletion_date = datetime.now() + timedelta(days=retention_days)
        
        await db.execute("""
            INSERT INTO scheduled_deletions 
            (user_id, data_type, scheduled_date)
            VALUES (?, ?, ?)
        """, (user_id, data_type, deletion_date))
    
    async def execute_deletion_job(self):
        """Periodic job to delete expired data"""
        due_deletions = await db.query("""
            SELECT user_id, data_type FROM scheduled_deletions
            WHERE scheduled_date <= NOW()
        """)
        
        for user_id, data_type in due_deletions:
            await self._delete_user_data(user_id, data_type)
            await log_deletion(user_id, data_type)
```

**6. Integrity and Confidentiality**
```python
# Data protection controls
class DataSecurity:
    async def store_sensitive_data(self, user_id: str, data: str, data_type: str):
        """Encrypt and store sensitive data"""
        
        # Encrypt
        encrypted = self._encrypt_aes_256(data, user_id)
        
        # Hash field for indexing without exposing data
        field_hash = self._hash_sha256(data)
        
        # Store with audit trail
        await db.execute("""
            INSERT INTO encrypted_data 
            (user_id, data_type, encrypted_value, field_hash, created_at)
            VALUES (?, ?, ?, ?, ?)
        """, (user_id, data_type, encrypted, field_hash, datetime.now()))
        
        # Log access
        await log_data_access(
            user_id=user_id,
            data_type=data_type,
            action='stored',
            timestamp=datetime.now()
        )
    
    def _encrypt_aes_256(self, plaintext: str, key_id: str) -> str:
        """AES-256-GCM encryption"""
        key = self._get_encryption_key(key_id)
        cipher = AES.new(key, AES.MODE_GCM)
        ciphertext, tag = cipher.encrypt_and_digest(plaintext.encode())
        
        # Return nonce + tag + ciphertext
        return base64.b64encode(cipher.nonce + tag + ciphertext).decode()
```

**7. Accountability**
```python
# Demonstrate compliance
class Accountability:
    async def generate_dpia_report(self, processing_activity: str) -> dict:
        """Data Protection Impact Assessment"""
        
        dpia = {
            'processing_activity': processing_activity,
            'description': self._get_activity_description(processing_activity),
            'data_types': await self._get_data_types(processing_activity),
            'purpose': await self._get_purposes(processing_activity),
            'legal_basis': await self._get_legal_basis(processing_activity),
            'risks': await self._assess_risks(processing_activity),
            'mitigations': await self._get_mitigations(processing_activity),
            'assessment_date': datetime.now()
        }
        
        await self._store_dpia(dpia)
        return dpia
    
    async def generate_dsr_response(self, user_id: str, request_type: str) -> dict:
        """Respond to Data Subject Requests (DSR)"""
        
        if request_type == 'access':
            # SAR: Subject Access Request
            user_data = await self._compile_user_data(user_id)
            return {
                'request_type': 'access',
                'data': user_data,
                'response_date': datetime.now()
            }
        
        elif request_type == 'rectification':
            # Provide correction form
            return {'status': 'awaiting_user_input'}
        
        elif request_type == 'deletion':
            # Right to be forgotten
            await self._initiate_deletion(user_id)
            return {'status': 'deletion_scheduled'}
        
        elif request_type == 'data_portability':
            # Export in machine-readable format
            export_file = await self._export_user_data(user_id)
            return {'export_url': export_file}
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. GDPR Implementation Checklist

### Before Launch
- [ ] Conduct Data Protection Impact Assessment (DPIA)
- [ ] Document legal basis for data collection
- [ ] Implement Privacy by Design
- [ ] Create Data Processing Agreement (DPA) with vendors
- [ ] Implement encryption for all sensitive data
- [ ] Set up access controls and audit logging
- [ ] Create data retention schedules
- [ ] Document data flows and storage locations

### During Operations
- [ ] Implement automated expiration/deletion
- [ ] Monitor and log all data access
- [ ] Provide easy data access to users
- [ ] Handle Data Subject Requests within 30 days
- [ ] Maintain incident response procedures
- [ ] Regular security assessments
- [ ] Staff privacy training

### Incident Response
```python
class IncidentResponse:
    async def detect_and_respond_to_breach(self, incident: dict):
        """GDPR requires notification within 72 hours"""
        
        # 1. Assess severity
        affected_records = incident['records_involved']
        personal_data_types = incident['data_types']
        risk_level = self._assess_risk(affected_records, personal_data_types)
        
        # 2. Notify supervisory authority if high risk
        if risk_level == 'high':
            await self._notify_authority(
                incident=incident,
                deadline=datetime.now() + timedelta(hours=72)
            )
        
        # 3. Notify affected individuals if high risk
        if risk_level == 'high':
            await self._notify_individuals(
                affected_records,
                incident_details=incident
            )
        
        # 4. Document incident
        await self._log_incident(incident)
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. Privacy-Preserving Techniques

### Differential Privacy
```python
# Add noise to protect individual privacy in aggregates
import numpy as np

def add_differential_privacy(data: List[float], epsilon: float = 1.0):
    """Add noise for differential privacy"""
    sensitivity = 1.0  # Magnitude of change if one record changed
    scale = sensitivity / epsilon
    
    noise = np.random.laplace(0, scale, len(data))
    private_data = [d + n for d, n in zip(data, noise)]
    
    return private_data

# Example: Hide individual salary but maintain aggregate statistics
salaries = [50000, 60000, 75000, 80000, 95000]
private_salaries = add_differential_privacy(salaries, epsilon=1.0)
# Now: [49923, 60145, 74988, 80067, 95021]
# Individual salaries hidden, aggregate close to original
```

### Federated Learning
```python
# Train models without centralizing personal data
class FederatedTraining:
    async def train_with_user_data_local(self, user_id: str):
        """User trains model locally, sends only weights"""
        
        # 1. Download global model
        global_weights = await get_global_model()
        
        # 2. Train locally on user's device (data never leaves device)
        local_weights = await train_locally(
            model=global_weights,
            local_data_path='/user/local/data',
            epochs=5
        )
        
        # 3. Upload only weights, not data
        await upload_model_updates(local_weights)
        
        # 4. Server aggregates weights from all users
        updated_global = await aggregate_weights([
            user1_weights,
            user2_weights,
            user3_weights
        ])
        
        return updated_global
```

### Data Anonymization
```python
def anonymize_user_data(user_record: dict) -> dict:
    """K-anonymity: ensure record is not unique"""
    
    # Suppress quasi-identifiers
    anonymized = {
        'age_range': anonymize_age(user_record['age']),  # "30-39"
        'location': anonymize_location(user_record['zip']),  # "123**"
        'gender': user_record['gender'],
        'data': 'kept'
    }
    
    # Remove direct identifiers
    # No name, email, phone, SSN
    
    return anonymized
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 4. User Rights Implementation

### Subject Access Request (SAR)
```python
@app.post("/api/privacy/data-access-request")
async def request_data_access(user_id: str = Depends(get_user_id)):
    """User requests all their data (GDPR Article 15)"""
    
    # Collect all data
    user_data = {
        'profile': await db.query("SELECT * FROM users WHERE id = ?", user_id),
        'activity': await db.query("SELECT * FROM activity_logs WHERE user_id = ?", user_id),
        'preferences': await db.query("SELECT * FROM preferences WHERE user_id = ?", user_id),
        'communications': await db.query("SELECT * FROM emails WHERE user_id = ?", user_id)
    }
    
    # Create exportable format
    export_file = await create_json_export(user_data)
    
    # Send download link (valid for 30 days)
    download_link = create_secure_download_link(export_file, expires_in=30*24*3600)
    
    return {'download_url': download_link, 'expires_in': 30}
```

### Right to Deletion (RTBF)
```python
@app.post("/api/privacy/delete-account")
async def request_account_deletion(user_id: str = Depends(get_user_id)):
    """User requests deletion (GDPR Article 17)"""
    
    # Schedule deletion (allow 30-day grace period)
    deletion_scheduled = datetime.now() + timedelta(days=30)
    
    await db.execute("""
        UPDATE users SET status = 'deletion_pending', 
        deletion_date = ? WHERE id = ?
    """, (deletion_scheduled, user_id))
    
    # Send confirmation email with option to cancel
    await send_deletion_confirmation_email(user_id, deletion_scheduled)
    
    return {
        'status': 'deletion_scheduled',
        'deletion_date': deletion_scheduled.isoformat(),
        'message': 'Your account will be deleted in 30 days. Check email to cancel.'
    }

async def execute_account_deletion_job():
    """Periodic job to process deletions"""
    due_for_deletion = await db.query("""
        SELECT id FROM users WHERE status = 'deletion_pending' 
        AND deletion_date <= NOW()
    """)
    
    for user_id in due_for_deletion:
        await permanently_delete_user(user_id)
```

### Right to Portability
```python
@app.post("/api/privacy/data-portability")
async def request_data_portability(user_id: str = Depends(get_user_id)):
    """User requests data in portable format"""
    
    user_data = await compile_user_data(user_id)
    
    # Export in standard format (JSON, CSV, etc.)
    export_formats = {
        'json': create_json_export(user_data),
        'csv': create_csv_export(user_data)
    }
    
    # Create download package
    package = await create_portable_package(export_formats, user_id)
    
    return {'formats': list(export_formats.keys()), 'download_url': package}
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 5. Privacy Policy & Consent

### Consent Management
```typescript
interface ConsentRecord {
  userId: string;
  consentType: string;  // 'marketing', 'analytics', 'newsletter'
  consent: boolean;
  timestamp: Date;
  ipAddress: string;
  userAgent: string;
}

async function recordConsent(record: ConsentRecord) {
  // Store with full audit trail
  await db.execute(
    `INSERT INTO consent_records 
     (user_id, type, consent, timestamp, ip, user_agent)
     VALUES (?, ?, ?, ?, ?, ?)`,
    [record.userId, record.consentType, record.consent, 
     record.timestamp, record.ipAddress, record.userAgent]
  );
  
  // Notify user
  await sendConsentConfirmationEmail(record.userId, record.consentType);
}

async function withdrawConsent(userId: string, consentType: string) {
  // Record withdrawal
  await recordConsent({
    userId,
    consentType,
    consent: false,
    timestamp: new Date(),
    ipAddress: getClientIp(),
    userAgent: getUserAgent()
  });
  
  // Stop related processing
  await pauseProcessing(userId, consentType);
}
```

### Transparent Privacy Policy
```markdown
# Privacy Policy

## Data We Collect
- Name, email, preferences (with consent)
- Usage analytics (without consent, anonymized)
- Payment information (via Stripe, encrypted)

## Why We Collect It
- Service delivery (contractual obligation)
- Improvement (legitimate interest)
- Marketing (with explicit consent)

## Your Rights
- **Access**: Download all data about you (SAR)
- **Deletion**: Request account deletion (RTBF)
- **Portability**: Get data in portable format
- **Correction**: Update inaccurate data
- **Objection**: Opt-out of processing

## How Long We Keep It
- Account data: Lifetime (can request deletion)
- Activity logs: 90 days
- Payment records: 7 years (tax legal requirement)
- Marketing consent: 1 year

## Contact
Privacy Officer: privacy@aria.example.com
Supervisory Authority: [Local DPA]
```
</MySCode.Cell>

<VSCode.Cell language="markdown">
## 6. International Regulations

### GDPR (EU/EEA)
- Applies to: Any processing of EU residents' data
- Fines: Up to €20M or 4% of global revenue
- Key: Consent, DPIA, DPA, Data Processor

### CCPA (California)
- Applies to: CA residents' data, revenue >$25M
- Rights: Access, deletion, opt-out, non-discrimination
- Fines: Up to $7,500 per violation

### PIPEDA (Canada)
- Applies to: Canadian residents' data
- Key: Consent, accountability, openness
- Fines: Up to $100K CAD per violation

### LEI (Brazil)
- Applies to: Brazilian residents' data
- Key: Transparency, reasonable practices
- Fines: Up to 2% of revenue

## Privacy First Checklist

- [ ] Privacy Impact Assessment completed
- [ ] Encryption implemented for sensitive data
- [ ] Audit logging enabled for all access
- [ ] Data retention policies automated
- [ ] User consent clearly documented
- [ ] DSR handling processes built
- [ ] Data vendor agreements in place
- [ ] Incident response plan documented
- [ ] Privacy training completed
- [ ] Regular security audits scheduled
</MySCode.Cell>
```